# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:

# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:

import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY is not set"
print("Environment OK")

Environment OK


In [3]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [4]:
# TODO
# Create a .env file in this folder (project/starter) with the following variables
# (copy .env.example from the repo root):
# OPENAI_API_KEY="voc-..."
# OPENAI_BASE_URL="https://openai.vocareum.com/v1"
# TAVILY_API_KEY="tvly-..."

### VectorDB Instance

In [5]:
# TODO: Instantiate your ChromaDB Client
# Keep this path: Part 2 loads the database from "chromadb" (inside project/starter)
chroma_client = chromadb.PersistentClient(path="chromadb")
print("Chroma client ready. Existing collections:", [c.name for c in chroma_client.list_collections()])

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Chroma client ready. Existing collections: []


### Collection

In [6]:
# TODO: Pick one embedding function

# Part 2 rebuilds this same embedding function when it loads the collection
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key_env_var="OPENAI_API_KEY",
    api_base=os.getenv("OPENAI_BASE_URL"),
)

In [7]:
# TODO: Create a collection

# Keep this name: Part 2 loads the collection "udaplay"
collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn,
)
print("Collection:", collection.name, "| documents so far:", collection.count())

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Collection: udaplay | documents so far: 0


In [8]:
class VectorStoreManager:
    """Reusable wrapper around a persistent ChromaDB collection."""

    def __init__(self, path="chromadb", collection_name="udaplay"):
        self.client = chromadb.PersistentClient(path=path)
        self.embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
            api_key_env_var="OPENAI_API_KEY",
            api_base=os.getenv("OPENAI_BASE_URL"),
        )
        self.collection = self.client.get_or_create_collection(
            name=collection_name,
            embedding_function=self.embedding_fn,
        )

    @staticmethod
    def game_to_text(game):
        """Text that gets embedded. Name/platform/year/genre/publisher are included
        so questions about any of them can match semantically."""
        return (
            f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - "
            f"Genre: {game['Genre']}. Publisher: {game['Publisher']}. {game['Description']}"
        )

    def load_games(self, data_dir="games"):
        """Load every JSON file in data_dir and upsert it (safe to re-run)."""
        ids, docs, metas = [], [], []
        for file_name in sorted(os.listdir(data_dir)):
            if not file_name.endswith(".json"):
                continue
            with open(os.path.join(data_dir, file_name), "r", encoding="utf-8") as f:
                game = json.load(f)
            ids.append(os.path.splitext(file_name)[0])   # file name (like 001) as ID
            docs.append(self.game_to_text(game))
            metas.append(game)
        # upsert (not add) so re-running the notebook does not create duplicate-ID errors
        self.collection.upsert(ids=ids, documents=docs, metadatas=metas)
        return len(ids)

    def search(self, query, n_results=3):
        """Semantic search. Returns a list of dicts with the game fields and the distance."""
        res = self.collection.query(query_texts=[query], n_results=n_results)
        out = []
        for i in range(len(res["ids"][0])):
            item = dict(res["metadatas"][0][i])
            item["id"] = res["ids"][0][i]
            item["distance"] = res["distances"][0][i]
            out.append(item)
        return out

    def count(self):
        return self.collection.count()

### Add documents

In [9]:
# Run this notebook from project/starter: the game files are in ./games
store = VectorStoreManager(path="chromadb", collection_name="udaplay")
n = store.load_games("games")
print(f"Processed {n} game files -> collection now holds {store.count()} documents")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Processed 15 game files -> collection now holds 15 documents


In [10]:

peek = store.collection.get(limit=15)
for _id, doc in zip(peek["ids"], peek["documents"]):
    print(_id, "|", doc[:140])
     

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


001 | [PlayStation 1] Gran Turismo (1997) - Genre: Racing. Publisher: Sony Computer Entertainment. A realistic racing simulator featuring a wide a
002 | [PlayStation 2] Grand Theft Auto: San Andreas (2004) - Genre: Action-adventure. Publisher: Rockstar Games. An expansive open-world game set 
003 | [PlayStation 3] Gran Turismo 5 (2010) - Genre: Racing. Publisher: Sony Computer Entertainment. A comprehensive racing simulator featuring a 
004 | [PlayStation 4] Marvel's Spider-Man (2018) - Genre: Action-adventure. Publisher: Sony Interactive Entertainment. An open-world superhero gam
005 | [PlayStation 5] Marvel's Spider-Man 2 (2023) - Genre: Action-adventure. Publisher: Sony Interactive Entertainment. The sequel to the acclaim
006 | [Game Boy Color] Pokémon Gold and Silver (1999) - Genre: Role-playing. Publisher: Nintendo. Second-generation Pokémon games introducing new 
007 | [Game Boy Advance] Pokémon Ruby and Sapphire (2002) - Genre: Role-playing. Publisher: Nintendo. Third-generation

In [12]:
test_queries = [
    "Who published Grand Theft Auto: San Andreas?",
    "When was Pokemon Gold and Silver released?",
    "What platform was Super Mario 64 launched on?",
    "a realistic racing simulator",
    "first 3D platformer from Nintendo",
]

for q in test_queries:
    print("=" * 90)
    print("QUERY:", q)
    for r in store.search(q, n_results=3):
        print(f"  dist={r['distance']:.3f} | {r['Name']} | {r['Platform']} | {r['YearOfRelease']} | {r['Publisher']}")

QUERY: Who published Grand Theft Auto: San Andreas?


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


  dist=0.246 | Grand Theft Auto: San Andreas | PlayStation 2 | 2004 | Rockstar Games
  dist=0.420 | Gran Turismo | PlayStation 1 | 1997 | Sony Computer Entertainment
  dist=0.427 | Gran Turismo 5 | PlayStation 3 | 2010 | Sony Computer Entertainment
QUERY: When was Pokemon Gold and Silver released?
  dist=0.239 | Pokémon Gold and Silver | Game Boy Color | 1999 | Nintendo
  dist=0.302 | Pokémon Ruby and Sapphire | Game Boy Advance | 2002 | Nintendo
  dist=0.452 | Super Mario 64 | Nintendo 64 | 1996 | Nintendo
QUERY: What platform was Super Mario 64 launched on?
  dist=0.216 | Super Mario 64 | Nintendo 64 | 1996 | Nintendo
  dist=0.301 | Super Mario World | Super Nintendo Entertainment System (SNES) | 1990 | Nintendo
  dist=0.410 | Mario Kart 8 Deluxe | Nintendo Switch | 2017 | Nintendo
QUERY: a realistic racing simulator
  dist=0.277 | Gran Turismo | PlayStation 1 | 1997 | Sony Computer Entertainment
  dist=0.290 | Gran Turismo 5 | PlayStation 3 | 2010 | Sony Computer Entertainment
  dis

In [11]:
# Run this notebook from project/starter: the game files are in ./games
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )

Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given
